In [5]:
!pip install wrds
!pip install xgboost
!pip install shap

In [6]:
import wrds
import pandas as pd
import numpy as np
from datetime import datetime

In [23]:
class AlphaPipeline:

    def __init__(self, years_back=5):
        self.db = wrds.Connection()
        self.years_back = years_back

    def _start_date(self):
        today = datetime.today()
        return today.replace(year=today.year - self.years_back).strftime('%Y-%m-%d')

    # -----------------------------------
    # CRSP
    # -----------------------------------
    def get_crsp(self):
        query = f"""
            SELECT permno, date, ret, retx, prc, vol, shrout
            FROM crsp.msf
            WHERE date >= '{self._start_date()}'
        """
        df = self.db.raw_sql(query)
        df['date'] = pd.to_datetime(df['date'])
        df['mktcap'] = df['prc'].abs() * df['shrout']
        return df

    # -----------------------------------
    # FF
    # -----------------------------------
    def get_ff(self):
        query = f"""
            SELECT date, mktrf, smb, hml, rmw, cma, rf
            FROM ff.fivefactors_monthly
            WHERE date >= '{self._start_date()}'
        """
        df = self.db.raw_sql(query)
        df['date'] = pd.to_datetime(df['date'])
        df[['mktrf','smb','hml','rmw','cma','rf']] /= 100
        return df

    # -----------------------------------
    # COMPUSTAT
    # -----------------------------------
    def get_compustat(self):
        query = f"""
            SELECT gvkey, datadate,
                   at, ceq, ni, sale,
                   ebit, oiadp,
                   dltt, dlc,
                   oancf,
                   prcc_f, csho
            FROM comp.funda
            WHERE datadate >= '{self._start_date()}'
        """
        df = self.db.raw_sql(query)
        df['datadate'] = pd.to_datetime(df['datadate'])
        return df

    # -----------------------------------
    # IBES
    # -----------------------------------
    def get_ibes(self):
        query = f"""
            SELECT ticker, statpers, meanest, stdev
            FROM ibes.statsum_epsus
            WHERE statpers >= '{self._start_date()}'
        """
        df = self.db.raw_sql(query)
        df['statpers'] = pd.to_datetime(df['statpers'])
        return df

    def build_analyst(self, ibes):
        ibes = ibes.sort_values(['ticker','statpers'])

        ibes['eps_revision'] = ibes['meanest'] - ibes.groupby('ticker')['meanest'].shift(1)
        ibes['eps_dispersion'] = ibes['stdev'] / (ibes['meanest'].abs() + 1e-6)

        return ibes

    # -----------------------------------
    # TECHNICAL (EXPANDED)
    # -----------------------------------
    def build_technical(self, df):

        df = df.sort_values(['permno','date'])

        # Momentum
        for w in [1,2,3,6,9,12]:
            df[f'mom_{w}'] = df.groupby('permno')['ret'].rolling(w).sum().reset_index(level=0,drop=True)

        # Volatility
        for w in [3,6,12]:
            df[f'vol_{w}'] = df.groupby('permno')['ret'].rolling(w).std().reset_index(level=0,drop=True)

        # Price deviation
        df['price_dev_3'] = df['prc'] / df.groupby('permno')['prc'].rolling(3).mean().reset_index(level=0,drop=True)
        df['price_dev_6'] = df['prc'] / df.groupby('permno')['prc'].rolling(6).mean().reset_index(level=0,drop=True)

        # Liquidity
        df['turnover'] = df['vol'] / df['shrout']
        df['vol_spike'] = df['vol'] / df.groupby('permno')['vol'].rolling(12).mean().reset_index(level=0,drop=True)

        # Advanced volatility
        df['down_vol'] = df['ret'].clip(upper=0).rolling(6).std()
        df['vol_ratio'] = df['vol_3'] / (df['vol_6'] + 1e-6)

        # Target
        df['target'] = df.groupby('permno')['ret'].shift(-1)

        return df

    # -----------------------------------
    # FUNDAMENTALS (EXPANDED)
    # -----------------------------------
    def build_fundamental(self, comp):

        comp = comp.sort_values(['gvkey','datadate'])

        # Profitability
        comp['roa'] = comp['ni'] / comp['at']
        comp['roe'] = comp['ni'] / comp['ceq']
        comp['ebit_ratio'] = comp['ebit'] / comp['at']

        # Growth
        comp['sales_growth'] = comp.groupby('gvkey')['sale'].pct_change(fill_method=None)
        comp['earnings_growth'] = comp.groupby('gvkey')['ni'].pct_change(fill_method=None)

        # Leverage
        comp['leverage'] = (comp['dltt'] + comp['dlc']) / comp['at']

        # Cash flow
        comp['cash_flow'] = comp['oancf'] / comp['at']

        # Valuation
        comp['book_to_price'] = comp['ceq'] / (comp['prcc_f'] * comp['csho'])
        comp['earnings_yield'] = comp['ni'] / (comp['prcc_f'] * comp['csho'])

        # Quality
        comp['accruals'] = (comp['ni'] - comp['oancf']) / comp['at']

        return comp

    # -----------------------------------
    # LINK
    # -----------------------------------
    def get_link(self):
        query = """
            SELECT gvkey, lpermno AS permno, linkdt, linkenddt
            FROM crsp.ccmxpf_linktable
            WHERE linktype IN ('LU','LC')
        """
        df = self.db.raw_sql(query)
        df['linkdt'] = pd.to_datetime(df['linkdt'])
        df['linkenddt'] = pd.to_datetime(df['linkenddt'])
        return df

    # -----------------------------------
    # MERGE
    # -----------------------------------
    def merge_all(self, crsp, ff, comp, link, ibes=None):

        crsp['ym'] = crsp['date'].dt.to_period('M')
        ff['ym'] = ff['date'].dt.to_period('M')

        df = crsp.merge(ff.drop(columns=['date']), on='ym', how='left')

        # Compustat merge
        comp = comp.merge(link, on='gvkey')
        comp = comp[
            (comp['datadate'] >= comp['linkdt']) &
            (comp['datadate'] <= comp['linkenddt'])
        ]

        comp['ym'] = comp['datadate'].dt.to_period('M')

        fund_cols = [
            'roa','roe','ebit_ratio','sales_growth','earnings_growth',
            'leverage','cash_flow',
            'book_to_price','earnings_yield','accruals'
        ]

        df = df.merge(comp[['permno','ym'] + fund_cols],
                      on=['permno','ym'], how='left')

        # Apply lag (CRITICAL)
        for col in fund_cols:
            df[col] = df.groupby('permno')[col].shift(3)

        # IBES safe merge
        if ibes is not None:
            ibes['ym'] = ibes['statpers'].dt.to_period('M')
            ibes_agg = ibes.groupby('ym')[['eps_revision','eps_dispersion']].mean().reset_index()
            df = df.merge(ibes_agg, on='ym', how='left')

        df = df.sort_values(['permno','date'])

        df = df.drop(columns=['ym'])

        return df

    # -----------------------------------
    # EXPAND FEATURES
    # -----------------------------------
    def expand(self, df):

        df = df.sort_values(['permno','date'])

        base_cols = [c for c in df.columns if c not in ['permno','date','target']]

        # Winsorize first
        for col in base_cols:
            df[col] = df.groupby('date')[col].transform(
                lambda x: x.clip(x.quantile(0.01), x.quantile(0.99))
            )

        # Rank
        for col in base_cols:
            df[f'{col}_r'] = df.groupby('date')[col].rank(pct=True)

        # Z-score
        for col in base_cols:
            df[f'{col}_z'] = df.groupby('date')[col].transform(
                lambda x: (x - x.mean()) / (x.std() + 1e-8)
            )

        # Interactions (HIGH VALUE)
        if 'roa' in df.columns and 'mom_3' in df.columns:
            df['roa_mom'] = df['roa'] * df['mom_3']

        if 'vol_3' in df.columns and 'mom_3' in df.columns:
            df['vol_mom'] = df['vol_3'] * df['mom_3']

        if 'book_to_price' in df.columns and 'mom_3' in df.columns:
            df['value_mom'] = df['book_to_price'] * df['mom_3']

        return df

    # -----------------------------------
    # CLEAN FEATURES
    # -----------------------------------
    def clean_features(self, df):

        drop_cols = ['permno', 'date', 'target']
        features = [c for c in df.columns if c not in drop_cols]

        nunique = df[features].nunique()
        features = nunique[nunique > 10].index.tolist()

        df[features] = df[features].fillna(0)

        return df, features

    # -----------------------------------
    # RUN
    # -----------------------------------
    def run(self):

        crsp = self.get_crsp()
        crsp = self.build_technical(crsp)

        ff = self.get_ff()

        comp = self.get_compustat()
        comp = self.build_fundamental(comp)

        link = self.get_link()

        ibes = self.get_ibes()
        ibes = self.build_analyst(ibes)

        df = self.merge_all(crsp, ff, comp, link, ibes)

        # Fill missing
        num_cols = df.select_dtypes(include=np.number).columns
        df[num_cols] = df.groupby('permno')[num_cols].ffill(limit=12)
        df[num_cols] = df.groupby('permno')[num_cols].bfill()

        df = df[df['target'].notna()]
        df = df.dropna(subset=['mktrf'])

        df = self.expand(df)

        df, features = self.clean_features(df)

        # Memory optimization
        num_cols = df.select_dtypes(include=np.number).columns
        df[num_cols] = df[num_cols].astype('float32')

        print("FINAL SHAPE:", df.shape)
        print("NUM FEATURES:", len(features))

        return df

    def close(self):
        self.db.close()

In [24]:
# -----------------------------------
# RUN
# -----------------------------------
if __name__ == "__main__":

    pipe = AlphaPipeline(years_back=5)

    # Run pipeline
    dataset = pipe.run()

    print("\nFinal shape:", dataset.shape)
    print(dataset.head())

    # -----------------------------------
    # FILE NAMING (VERSIONED)
    # -----------------------------------
    from datetime import datetime
    ts = datetime.now().strftime("%Y%m%d")

    parquet_name = f"ml_factor_dataset_{ts}.parquet"

    # -----------------------------------
    # SAVE LOCALLY (FAST + COMPRESSED)
    # -----------------------------------
    dataset.to_parquet(
        parquet_name,
        index=False,
        compression="snappy"
    )

    print(f"Saved locally: {parquet_name}")

    # -----------------------------------
    # SAVE TO GOOGLE DRIVE
    # -----------------------------------
    try:
        from google.colab import drive
        import os

        # Mount only if not already mounted
        if not os.path.exists("/content/drive"):
            drive.mount('/content/drive')

        drive_path = f"/content/drive/MyDrive/{parquet_name}"

        dataset.to_parquet(
            drive_path,
            index=False,
            compression="snappy"
        )

        print(f"Saved to Drive: {drive_path}")

    except Exception as e:
        print("Drive save skipped / failed:", e)

    # -----------------------------------
    # OPTIONAL: QUICK VALIDATION
    # -----------------------------------
    print("\nDataset summary:")
    print(dataset.describe().iloc[:5])

    pipe.close()

Enter your WRDS username [root]:vishw045
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
FINAL SHAPE: (438654, 123)
NUM FEATURES: 118

Final shape: (438654, 123)
    permno       date       ret      retx         prc      vol   shrout  \
0  10026.0 2021-03-31 -0.007275 -0.010897  157.029999  29518.0  19034.0   
1  10026.0 2021-04-30  0.048271  0.048271  164.610001  24625.0  19036.0   
2  10026.0 2021-05-28  0.066642  0.066642  175.580002  13818.0  19036.0   
3  10026.0 2021-06-30 -0.003058 -0.006664  174.410004  22259.0  19061.0   
4  10026.0 2021-07-30 -0.057508 -0.057508  164.380005  13204.0  19064.0   

       mktcap     mom_1     mom_2  ...  leverage_z  cash_flow_z  \
0  2988909.00 -0.007275  0.040996  ...         0.0          0.0   
1  3133516.00  0.048271  0.040996  ...     

# Model Training

In [78]:
import pandas as pd
import numpy as np

# ✅ Use parquet
df = pd.read_parquet("/content/drive/MyDrive/ml_factor_dataset_20260331.parquet")

print(df.shape)
df.head()

(438654, 123)


,permno,date,ret,retx,prc,vol,shrout,mktcap,mom_1,mom_2,...,leverage_z,cash_flow_z,book_to_price_z,earnings_yield_z,accruals_z,eps_revision_z,eps_dispersion_z,roa_mom,vol_mom,value_mom
0,10026.0,2021-03-31,-0.007275,-0.010897,157.029999,29518.0,19034.0,2988909.00,-0.007275,0.040996,...,0.0,0.0,0.0,0.0,0.0,0.000000,-0.000006,0.0,0.004142,0.0
1,10026.0,2021-04-30,0.048271,0.048271,164.610001,24625.0,19036.0,3133516.00,0.048271,0.040996,...,0.0,0.0,0.0,0.0,0.0,0.000000,-0.000006,0.0,0.004142,0.0
2,10026.0,2021-05-28,0.066642,0.066642,175.580002,13818.0,19036.0,3342341.00,0.066642,0.114913,...,0.0,0.0,0.0,0.0,0.0,-0.011508,0.000023,0.0,0.004142,0.0
3,10026.0,2021-06-30,-0.003058,-0.006664,174.410004,22259.0,19061.0,3324429.00,-0.003058,0.063584,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.004041,0.0
4,10026.0,2021-07-30,-0.057508,-0.057508,164.380005,13204.0,19064.0,3133740.25,-0.057508,-0.060566,...,0.0,0.0,0.0,0.0,0.0,0.005787,0.000000,0.0,0.000378,0.0


In [79]:
# Cross-sectional target (THIS IS KEY)
df['target_rank'] = df.groupby('date')['target'].rank(pct=True)

# Drop raw target
df = df.drop(columns=['target'])

In [80]:
exclude = ['permno', 'date', 'target_rank']
features = [c for c in df.columns if c not in exclude]

df_clean = df.copy()

# Handle inf
df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill missing
df_clean[features] = df_clean[features].fillna(0)

X = df_clean[features]
y = df_clean['target_rank']

print("Feature shape:", X.shape)

Feature shape: (438654, 120)


In [81]:
df_clean = df_clean.sort_values('date')

dates = df_clean['date'].unique()
split_date = dates[int(len(dates) * 0.7)]

train = df_clean[df_clean['date'] < split_date]
test  = df_clean[df_clean['date'] >= split_date]

X_train = train[features]
y_train = train['target_rank']

X_test = test[features]
y_test = test['target_rank']

In [82]:
import xgboost as xgb

model = xgb.XGBRegressor(
    tree_method='hist',
    max_depth=7,
    n_estimators=300,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.6
)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [83]:
feat_imp = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

print(feat_imp.head(25))

              feature  importance
89           mom_12_z    0.053136
104              rf_z    0.033318
38     eps_dispersion    0.028863
62              hml_r    0.027405
11             mom_12    0.025780
26                 rf    0.021009
37       eps_revision    0.020168
50           mom_12_r    0.019816
60            mktrf_r    0.019542
91            vol_6_z    0.018959
102             rmw_z    0.018224
64              cma_r    0.017810
61              smb_r    0.016805
21              mktrf    0.016667
25                cma    0.016608
115    eps_revision_z    0.016540
22                smb    0.015557
116  eps_dispersion_z    0.015349
14             vol_12    0.014367
92           vol_12_z    0.014214
103             cma_z    0.013866
53           vol_12_r    0.013824
99            mktrf_z    0.013695
84            mom_1_z    0.013497
97         down_vol_z    0.012499


In [84]:
# -----------------------------------
# SIGNAL SCORING (ROBUST VERSION)
# -----------------------------------
corr = df_clean[features + ['target_rank']].corr()['target_rank']

# Map correlation to feature importance table
feat_imp['corr'] = feat_imp['feature'].map(corr)

# Direction (sign of relationship)
feat_imp['direction'] = np.sign(feat_imp['corr'])

# Rank importance (0 → 1)
feat_imp['imp_rank'] = feat_imp['importance'].rank(pct=True)

# Rank correlation strength using ABS
feat_imp['corr_rank'] = feat_imp['corr'].abs().rank(pct=True)

# Combine ranks (multiplicative)
feat_imp['signal_strength'] = feat_imp['imp_rank'] * feat_imp['corr_rank']

# Sort signals
signals = feat_imp.sort_values(by='signal_strength', ascending=False)

print(signals[['feature','importance','corr','direction','signal_strength']].head(25))

          feature  importance      corr  direction  signal_strength
89       mom_12_z    0.053136  0.151091        1.0         0.983193
11         mom_12    0.025780  0.144812        1.0         0.934174
91        vol_6_z    0.018959 -0.136863       -1.0         0.878361
50       mom_12_r    0.019816  0.128479        1.0         0.846709
92       vol_12_z    0.014214 -0.140233       -1.0         0.806303
97     down_vol_z    0.012499 -0.161011       -1.0         0.800000
14         vol_12    0.014367 -0.135380       -1.0         0.792857
53       vol_12_r    0.013824 -0.132917       -1.0         0.755672
88        mom_9_z    0.011117  0.135531        1.0         0.705882
49        mom_9_r    0.011683  0.120680        1.0         0.670028
90        vol_3_z    0.011493 -0.120672       -1.0         0.656373
52        vol_6_r    0.008582 -0.134122       -1.0         0.623950
54  price_dev_3_r    0.011706  0.066369        1.0         0.605672
10          mom_9    0.008396  0.126127        1

Retrain test etc

In [85]:
top_feats = signals.head(25).copy()
top_features = top_feats['feature'].tolist()

In [86]:
alpha_train = train[['permno','date']].copy()
alpha_test  = test[['permno','date']].copy()

for _, row in top_feats.iterrows():
    f = row['feature']
    d = row['direction']

    # Cross-sectional rank signal
    alpha_train[f'alpha_{f}'] = d * train.groupby('date')[f].rank(pct=True)
    alpha_test[f'alpha_{f}']  = d * test.groupby('date')[f].rank(pct=True)

In [87]:
weights = top_feats.set_index('feature')['signal_strength']

for f in top_features:
    alpha_train[f'alpha_{f}'] *= weights[f]
    alpha_test[f'alpha_{f}']  *= weights[f]

In [88]:
alpha_cols = [c for c in alpha_train.columns if c.startswith('alpha_')]

alpha_train['alpha'] = alpha_train[alpha_cols].sum(axis=1)
alpha_test['alpha']  = alpha_test[alpha_cols].sum(axis=1)

In [89]:
import xgboost as xgb

model2 = xgb.XGBRegressor(
    tree_method='hist',
    max_depth=6,
    n_estimators=300,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.6
)

model2.fit(alpha_train[alpha_cols], y_train)

alpha_test['alpha'] = model2.predict(alpha_test[alpha_cols])

In [90]:
alpha_test['alpha_rank'] = alpha_test.groupby('date')['alpha'].rank(pct=True)

In [91]:
daily_ic = alpha_test.groupby('date').apply(
    lambda x: np.corrcoef(x['alpha'], test.loc[x.index, 'target_rank'])[0,1]
)

print("Mean IC:", daily_ic.mean())

Mean IC: 0.14002214843573652


/tmp/ipykernel_22061/4039420629.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_ic = alpha_test.groupby('date').apply(


In [92]:
merged = test.copy()
merged['alpha_rank'] = alpha_test['alpha_rank']

In [93]:
long = merged[merged['alpha_rank'] > 0.9]
short = merged[merged['alpha_rank'] < 0.1]

# ✅ USE REAL RETURNS
long_ret = long.groupby('date')['target_rank'].mean()
short_ret = short.groupby('date')['target_rank'].mean()

strategy = long_ret - short_ret

print("Mean return:", strategy.mean())
print("Sharpe:", strategy.mean() / (strategy.std() + 1e-8))

Mean return: 0.1321133516982542
Sharpe: 1.1452361600780372


Factors

In [95]:
mom_cols = [
    'mom_12_z','mom_9_z','mom_6_z','mom_3_z','mom_1_z'
]

alpha_test['alpha_momentum'] = df_clean[mom_cols].mean(axis=1)

In [97]:
vol_cols = [
    'vol_12_z','vol_6_z','vol_3_z','down_vol_z'
]

alpha_test['alpha_lowvol'] = -df_clean[vol_cols].mean(axis=1)

In [148]:
alpha_test['alpha_trend'] = (
    df_clean['mom_12_z'] - df_clean['vol_12_z']
)

In [100]:
alpha_test['alpha_price_dev'] = df_clean[
    ['price_dev_3_r','price_dev_6_r']
].mean(axis=1)

In [101]:
alpha_test['alpha_short'] = df_clean[
    ['ret_z','retx_z']
].mean(axis=1)

In [102]:
for col in alpha_test.columns:
    if col not in ['permno','date']:
        alpha_test[col] = alpha_test.groupby('date')[col].rank(pct=True)

In [103]:
alpha_cols = [c for c in alpha_test.columns if c.startswith('alpha_')]

alpha_test['alpha_combo'] = alpha_test[alpha_cols].mean(axis=1)

In [149]:
weights = {
    'alpha_momentum': 0.0,
    'alpha_lowvol': 0.0,
    'alpha_trend': 1,
    'alpha_price_dev': 0.0,
    'alpha_short': 0.0
}

alpha_test['alpha_combo'] = sum(
    alpha_test[k] * v for k,v in weights.items()
)

In [150]:
alpha_test['alpha_rank'] = alpha_test.groupby('date')['alpha_combo'].rank(pct=True)

In [151]:
merged = df_clean.merge(alpha_test[['permno','date','alpha_rank']], on=['permno','date'])

long = merged[merged['alpha_rank'] > 0.9]
short = merged[merged['alpha_rank'] < 0.1]

long_ret = long.groupby('date')['target_rank'].mean()
short_ret = short.groupby('date')['target_rank'].mean()

strategy = long_ret - short_ret

print("Sharpe:", strategy.mean() / strategy.std())

Sharpe: 1.2226639483388957
